<a href="https://colab.research.google.com/github/MZiaAfzal71/Average_Weighted_Path_Vector/blob/main/Data%20Files/Models/XGBoostModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/MZiaAfzal71/Average_Weighted_Path_Vector.git
%cd Average_Weighted_Path_Vector/Data\ Files

Cloning into 'Average_Weighted_Path_Vector'...
remote: Enumerating objects: 768, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (173/173), done.
remote: Total 768 (delta 112), reused 8 (delta 8), pack-reused 585 (from 1)
Receiving objects: 100% (768/768), 30.81 MiB | 31.24 MiB/s, done.
Resolving deltas: 100% (266/266), done.
/content/Average_Weighted_Path_Vector/Data Files


In [2]:
!pip install osfclient rdkit
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 52.3 MB/s eta 0:00:00


In [3]:
from osfclient.api import OSF
from subprocess import run

# Replace with your OSF project ID
project_id = "p5ga2"   # e.g. from https://osf.io/abcd3/
osf = OSF()
project = osf.project(project_id)
store = project.storage("osfstorage")

desc_folder = []
for fold in store.folders:
    if fold.path.strip("/") == "Descriptors Data":
        desc_folder = fold
        break

# Download all files and keep folder structure
for f in desc_folder.files:
    local_path = f.path.strip("/")            # keep folders
    local_dir = os.path.dirname(local_path)   # extract dir
    if local_dir and not os.path.exists(local_dir):
        os.makedirs(local_dir, exist_ok=True) # create dirs if missing
    with open(local_path, "wb") as out:
        f.write_to(out)
    if local_path.endswith(".zip"):
      command = f"unzip '{local_path}' -d '{local_dir}'"
      run(command, shell=True)
      print(f"\nUnzipped {local_path} -> {local_dir}")
      continue
    print(f"Downloaded {f.path} -> {local_path}")

100%|██████████| 23.8M/23.8M [00:00<00:00, 104Mbytes/s] 



Unzipped Descriptors Data/Descriptors Data.zip -> Descriptors Data


100%|██████████| 13.5M/13.5M [00:00<00:00, 97.9Mbytes/s]



Unzipped Descriptors Data/RAW_Descriptors_PWAV.zip -> Descriptors Data


In [4]:
# from xgboost import XGBRegressor # The model to learn Fusion points of Alkanes
# from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error  # To find errors
# from sklearn.preprocessing import StandardScaler
# import numpy as np
# import pandas as pd

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupShuffleSplit
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

import numpy as np
import pandas as pd
import json

In [21]:
# === Global Config ===
input_dir = "Descriptors Data"
output_dir = "XGBoost Results"
os.makedirs(output_dir, exist_ok=True)

property_sheets = ["Log VP", "MP", "BP", "LogBCF", "LogS", "LogP"]
desc_keys = ["MACCS", "Morgan", "PWAV_raw"]   # updated names

SPLIT_MODE = "scaffold"   # options: "benchmark", "scaffold"
TEST_SIZE = 0.2
RANDOM_STATE = 42
N_PCA = 64

SAVE_ALL_PREDICTIONS = True

def safe_name(x: str) -> str:
    return str(x).replace(" ", "").replace("/", "_").replace("\\", "_")

def murcko_scaffold(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        return None

def get_split_indices(df, split_mode="benchmark", test_size=0.2, random_state=42):
    if split_mode == "benchmark":
        train_idx = df.index[df["Training/Test"] == "Training"].tolist()
        test_idx = df.index[df["Training/Test"] == "Test"].tolist()
        split_info = df[["SMILES", "Training/Test"]].copy()
        split_info["split_mode"] = "benchmark"
        return train_idx, test_idx, split_info

    elif split_mode == "scaffold":
        if "SMILES" not in df.columns:
            raise ValueError("SMILES column is required for scaffold split.")

        scaffold_df = df[["SMILES"]].copy()
        scaffold_df["scaffold"] = scaffold_df["SMILES"].apply(murcko_scaffold)

        valid_mask = scaffold_df["scaffold"].notna()
        if valid_mask.sum() == 0:
            raise ValueError("No valid scaffolds generated.")

        valid_idx = scaffold_df.index[valid_mask]
        groups = scaffold_df.loc[valid_idx, "scaffold"]

        gss = GroupShuffleSplit(
            n_splits=1,
            test_size=test_size,
            random_state=random_state
        )
        train_pos, test_pos = next(gss.split(valid_idx, groups=groups))

        train_idx = valid_idx[train_pos].tolist()
        test_idx = valid_idx[test_pos].tolist()

        split_info = scaffold_df.copy()
        split_info["split"] = "unused"
        split_info.loc[train_idx, "split"] = "Training"
        split_info.loc[test_idx, "split"] = "Test"
        split_info["split_mode"] = "scaffold"

        return train_idx, test_idx, split_info

    else:
        raise ValueError(f"Unknown split_mode: {split_mode}")

def get_feature_columns(df, prop, desc_key):
    # Target column
    target_col = f"{prop}-Measured"

    meta_cols = {
        "Preferred_name", "SMILES", "Training/Test", f"{prop}-EPISuite Prediction",
        f"{prop}-Prediction from our model", "CAS RN",	"NAME",	"IUPAC Name",
        target_col
    }

    all_cols = list(df.columns)

    if desc_key == "PWAV_raw":
        pwav_cols = [c for c in all_cols if c.startswith(f"{prop}_PWAV_")]
        extra_cols = [c for c in all_cols if c.startswith(f"{prop}_EXTRA_")]
        return {
            "target_col": target_col,
            "pwav_cols": pwav_cols,
            "extra_cols": extra_cols,
            "direct_feature_cols": None
        }

    else:
        # For MACCS / Morgan / other ready-made descriptor tables
        direct_feature_cols = [c for c in all_cols if c not in meta_cols]
        return {
            "target_col": target_col,
            "pwav_cols": None,
            "extra_cols": None,
            "direct_feature_cols": direct_feature_cols
        }

def prepare_xy(df, prop, desc_key, train_idx, test_idx, n_pca=64):
    col_info = get_feature_columns(df, prop, desc_key)
    target_col = col_info["target_col"]

    train_df = df.loc[train_idx].copy()
    test_df = df.loc[test_idx].copy()

    y_train = train_df[target_col].to_numpy()
    y_test = test_df[target_col].to_numpy()

    pca_model = None

    if desc_key == "PWAV_raw":
        pwav_cols = col_info["pwav_cols"]
        extra_cols = col_info["extra_cols"]

        X_train_core = train_df[pwav_cols].to_numpy()
        X_test_core = test_df[pwav_cols].to_numpy()

        if n_pca is not None and n_pca < X_train_core.shape[1]:
            pca_model = PCA(n_components=n_pca, random_state=RANDOM_STATE)
            X_train_core = pca_model.fit_transform(X_train_core)
            X_test_core = pca_model.transform(X_test_core)

        if extra_cols:
            X_train_extra = train_df[extra_cols].to_numpy()
            X_test_extra = test_df[extra_cols].to_numpy()
            X_train = np.hstack([X_train_core, X_train_extra])
            X_test = np.hstack([X_test_core, X_test_extra])
        else:
            X_train = X_train_core
            X_test = X_test_core

    else:
        direct_cols = col_info["direct_feature_cols"]
        X_train = train_df[direct_cols].to_numpy()
        X_test = test_df[direct_cols].to_numpy()

    return X_train, X_test, y_train, y_test, train_df, test_df, pca_model



In [22]:
metrics_rows = []

for prop in property_sheets:
    safe_prop = safe_name(prop)

    for dsk in desc_keys:
        input_file = os.path.join(input_dir, f"{safe_prop}_{dsk}.parquet")
        print(f"\n🔹 Processing: {input_file}")

        if not os.path.exists(input_file):
            print(f"⚠️ File not found, skipping: {input_file}")
            continue

        df = pd.read_parquet(input_file)

        # Split
        train_idx, test_idx, split_info = get_split_indices(
            df,
            split_mode=SPLIT_MODE,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE
        )

        # Prepare arrays
        X_train, X_test, y_train, y_test, train_df, test_df, pca_model = prepare_xy(
            df,
            prop,
            dsk,
            train_idx,
            test_idx,
            n_pca=N_PCA
        )

        # Define model
        model = XGBRegressor(random_state=RANDOM_STATE)

        # Train
        model.fit(X_train, y_train)

        # Predict on test set
        y_pred_test = model.predict(X_test)

        # Metrics
        mae = mean_absolute_error(y_test, y_pred_test)
        rmse = root_mean_squared_error(y_test, y_pred_test)
        r2 = r2_score(y_test, y_pred_test)

        print(f"   ➤ R²   : {r2:.4f}")
        print(f"   ➤ MAE  : {mae:.4f}")
        print(f"   ➤ RMSE : {rmse:.4f}")

        metrics_rows.append({
            "property": prop,
            "descriptor": dsk,
            "split_mode": SPLIT_MODE,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "mae": mae,
            "rmse": rmse,
            "r2": r2
        })

        # Save split info
        split_file = os.path.join(
            output_dir,
            f"{safe_prop}_{dsk}_{SPLIT_MODE}_split.csv"
        )
        split_info.to_csv(split_file, index=False)

        # Save PCA metadata if used
        if pca_model is not None:
            pca_meta = {
                "property": prop,
                "descriptor": dsk,
                "split_mode": SPLIT_MODE,
                "n_components": int(pca_model.n_components_),
                "explained_variance_ratio_sum": float(np.sum(pca_model.explained_variance_ratio_))
            }
            pca_file = os.path.join(
                output_dir,
                f"{safe_prop}_{dsk}_{SPLIT_MODE}_pca.json"
            )
            with open(pca_file, "w") as f:
                json.dump(pca_meta, f, indent=2)

        # Save predictions on test set
        test_results = test_df[["Preferred_name", "SMILES"]].copy() if "Preferred_name" in test_df.columns else test_df[["SMILES"]].copy()
        test_results[f"{prop}-Measured"] = y_test
        test_results[f"{prop}_{dsk}_Prediction"] = y_pred_test
        test_results["split_mode"] = SPLIT_MODE

        pred_file = os.path.join(
            output_dir,
            f"{safe_prop}_{dsk}_{SPLIT_MODE}_test_predictions.xlsx"
        )
        test_results.to_excel(pred_file, index=False)

        # Optional: save predictions for all rows using train-fitted preprocessing
        if SAVE_ALL_PREDICTIONS:
            col_info = get_feature_columns(df, prop, dsk)

            if dsk == "PWAV_raw":
                pwav_cols = col_info["pwav_cols"]
                extra_cols = col_info["extra_cols"]

                X_all_core = df[pwav_cols].to_numpy()
                if pca_model is not None:
                    X_all_core = pca_model.transform(X_all_core)

                if extra_cols:
                    X_all_extra = df[extra_cols].to_numpy()
                    X_all = np.hstack([X_all_core, X_all_extra])
                else:
                    X_all = X_all_core
            else:
                X_all = df[col_info["direct_feature_cols"]].to_numpy()

            y_pred_all = model.predict(X_all)

            all_results = df[["Preferred_name", "SMILES"]].copy() if "Preferred_name" in df.columns else df[["SMILES"]].copy()
            target_col = f"{prop}-Measured"
            if target_col in df.columns:
                all_results[target_col] = df[target_col]
            if "Training/Test" in df.columns:
                all_results["Training/Test"] = df["Training/Test"]
            all_results[f"{prop}_{dsk}_Prediction"] = y_pred_all
            all_results["split_mode"] = SPLIT_MODE

            all_pred_file = os.path.join(
                output_dir,
                f"{safe_prop}_{dsk}_{SPLIT_MODE}_all_predictions.xlsx"
            )
            all_results.to_excel(all_pred_file, index=False)

        print(f"✅ Finished: {prop} | {dsk} | {SPLIT_MODE}")


🔹 Processing: Descriptors Data/LogVP_MACCS.parquet
   ➤ R²   : 0.5850
   ➤ MAE  : 1.4332
   ➤ RMSE : 2.0142
✅ Finished: Log VP | MACCS | scaffold

🔹 Processing: Descriptors Data/LogVP_Morgan.parquet
   ➤ R²   : 0.2630
   ➤ MAE  : 2.1233
   ➤ RMSE : 2.6840
✅ Finished: Log VP | Morgan | scaffold

🔹 Processing: Descriptors Data/LogVP_PWAV_raw.parquet
   ➤ R²   : 0.7382
   ➤ MAE  : 1.1703
   ➤ RMSE : 1.5995
✅ Finished: Log VP | PWAV_raw | scaffold

🔹 Processing: Descriptors Data/MP_MACCS.parquet
   ➤ R²   : 0.4996
   ➤ MAE  : 40.5727
   ➤ RMSE : 52.5531
✅ Finished: MP | MACCS | scaffold

🔹 Processing: Descriptors Data/MP_Morgan.parquet
   ➤ R²   : 0.3192
   ➤ MAE  : 47.2850
   ➤ RMSE : 61.2950
✅ Finished: MP | Morgan | scaffold

🔹 Processing: Descriptors Data/MP_PWAV_raw.parquet
   ➤ R²   : 0.5760
   ➤ MAE  : 36.3320
   ➤ RMSE : 48.3746
✅ Finished: MP | PWAV_raw | scaffold

🔹 Processing: Descriptors Data/BP_MACCS.parquet
   ➤ R²   : 0.4468
   ➤ MAE  : 44.9762
   ➤ RMSE : 60.9491
✅ Finishe

In [ ]:
# # === Paths & Settings ===
# input_dir = "Descriptors Data"
# output_dir = "XGBoost Results"
# os.makedirs(output_dir, exist_ok=True)

# property_sheets = ["Log VP", "MP", "BP", "LogBCF", "LogS", "LogP"]
# desc_keys = ["MACCS", "Morgan", "pwav"]

# # === Main Loop ===
# for prop in property_sheets:
#     for dsk in desc_keys:
#         # Input file
#         input_file = os.path.join(input_dir, f"{prop}_{dsk}.parquet")
#         print(f"\n🔹 Processing: {input_file}")

#         # Read data
#         df = pd.read_parquet(input_file)
#         prop_pred = f"{prop}-Measured"

#         # Feature matrix & target
#         X = df.iloc[:, 9:]
#         y = df[prop_pred]

#         # Train/test split
#         train_idx = df[df["Training/Test"] == "Training"].index.tolist()
#         test_idx  = df[df["Training/Test"] == "Test"].index.tolist()

#         X_train, X_valid = X.iloc[train_idx], X.iloc[test_idx]
#         y_train, y_valid = y.iloc[train_idx], y.iloc[test_idx]

#         # Define model
#         model = XGBRegressor(random_state=42)
#         # Alternative (your tuned version, uncomment if needed)
#         # model = XGBRegressor(
#         #     n_estimators=100, eta=0.081, max_depth=3, gamma=1, min_child_weight=1,
#         #     colsample_bytree=0.95, reg_lambda=1, reg_alpha=0, max_delta_step=0,
#         #     subsample=1, tree_method="hist", scale_pos_weight=1,
#         #     grow_policy="lossguide", max_leaves=0, max_bin=80,
#         #     num_parallel_tree=1, random_state=42
#         # )

#         # Train model
#         model.fit(X_train, y_train)

#         # Predict on validation set
#         y_pred = model.predict(X_valid)

#         # === Stats & Metrics ===
#         errors = abs(y_valid - y_pred)
#         stats_summary = errors.describe()

#         # Print metrics
#         print("📊 Error Summary:")
#         print(stats_summary)
#         print(f"   ➤ R²   : {r2_score(y_valid, y_pred):.4f}")
#         print(f"   ➤ MAE  : {mean_absolute_error(y_valid, y_pred):.4f}")
#         print(f"   ➤ RMSE : {root_mean_squared_error(y_valid, y_pred):.4f}")

#         # === Full Predictions ===
#         y_pred_all = model.predict(X)
#         df_results = df.iloc[:, :9].copy()
#         df_results[f"{prop} {dsk} Predictions"] = y_pred_all

#         # Save results
#         output_file = os.path.join(output_dir, f"{prop}_{dsk}_xgboost.xlsx")
#         df_results.to_excel(output_file, index=False)

#         print(f"✅ Results saved: {output_file}")

# print("\n🎉 All files have been processed successfully!")



🔹 Processing: Descriptors Data/Log VP_MACCS.parquet
📊 Error Summary:
count    679.000000
mean       1.039008
std        1.078119
min        0.004539
25%        0.292238
50%        0.689126
75%        1.399440
max        6.914137
Name: Log VP-Measured, dtype: float64
   ➤ R²   : 0.8228
   ➤ MAE  : 1.0390
   ➤ RMSE : 1.4967
✅ Results saved: XGBoost Results/Log VP_MACCS_xgboost.xlsx

🔹 Processing: Descriptors Data/Log VP_Morgan.parquet
📊 Error Summary:
count    679.000000
mean       1.346268
std        1.239952
min        0.002498
25%        0.445449
50%        0.969699
75%        1.970971
max        7.966365
Name: Log VP-Measured, dtype: float64
   ➤ R²   : 0.7352
   ➤ MAE  : 1.3463
   ➤ RMSE : 1.8297
✅ Results saved: XGBoost Results/Log VP_Morgan_xgboost.xlsx

🔹 Processing: Descriptors Data/Log VP_pwav.parquet
📊 Error Summary:
count    679.000000
mean       0.703033
std        0.802072
min        0.001165
25%        0.142756
50%        0.426437
75%        0.934681
max        5.525071
N